# Train the Rater — class-wise top-loss outer objective

This notebook trains **only the Rater** and saves a reusable `final_rater.pt`.
The final classifier remains a later experiment.

Representation and inner training are unchanged from the previous run:

- frozen **Waterbirds-ERM ResNet-50** 2048-D embeddings;
- two independent random-from-scratch `Linear(2048,2)` inner classifiers;
- every training sample receives a Rater-derived weight;
- each inner classifier sees the complete `train_no_aug` set once before every Rater update;
- 500 meta steps by default.

## New outer / Rater loss

The outer loss does **not** use Waterbirds group/background annotations.
It uses only the ordinary class label (`landbird` vs `waterbird`).

For each inner model, after its full inner epoch, the classifier is evaluated on
the **entire outer split** (`val_subset1`). For each class separately:

1. compute every example's cross-entropy loss;
2. keep the largest `OUTER_TOP_LOSS_PERCENT` percent losses within that class;
3. average those selected losses;
4. average the two class-tail means equally.

With the default 30%:

\[
L_{outer}
=\frac{1}{2}\left[
\operatorname{mean}(\operatorname{Top}_{30\%}(CE\mid y=0))
+
\operatorname{mean}(\operatorname{Top}_{30\%}(CE\mid y=1))
\right].
\]

So the Rater is optimized toward the hardest examples **within each class**,
without group annotations.

## New histogram layout

Each Rater histogram PNG is one 5-panel figure:

```text
Group 0 histogram  |                         |
Group 1 histogram  |  ALL GROUPS OVERLAID    |
Group 2 histogram  |      (larger panel)     |
Group 3 histogram  |                         |
```

The group labels are used **only for these diagnostic plots / WGA diagnostics**,
not for the outer meta-loss.

Saved to Drive:

```text
plots/
log.txt
summary.txt
final_rater.pt
```

In [1]:
# 1. Paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime
import os, sys, subprocess, shutil, re, torch

DRIVE_ROOT = Path("/content/drive/MyDrive/evidential_alignment_colab")
REPO_DIR = Path("/content/evidential_alignment")
DATA_ROOT = Path("/content/datasets")
WB_DIR = DATA_ROOT / "waterbird_complete95_forest2water2"
WB_ARCHIVE = DRIVE_ROOT / "archives" / "waterbird_complete95_forest2water2.tar.gz"

ERM_MODEL = DRIVE_ROOT / "erm_base_models" / "waterbirds_erm_resnet50_imagenet_100ep_best_val_acc.pt"
if not ERM_MODEL.exists():
    ERM_MODEL = DRIVE_ROOT / "erm_base_models" / "erm_waterbirds_resnet50_train_128B_100E_seed0_pretrained" / "best_val_acc_model.pt"
assert ERM_MODEL.exists(), ERM_MODEL

EMBED_ROOT = DRIVE_ROOT / "embeddings_waterbirds_erm"
OUTPUT_ROOT = DRIVE_ROOT / "rater_classwise_top_loss_runs"
LOCAL_PRE = Path("/content/rater_precompute_local")
LOCAL_RUN = Path("/content/rater_meta_only_local")

for p in [WB_ARCHIVE.parent, EMBED_ROOT, OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("ERM:", ERM_MODEL)
print("Embedding cache:", EMBED_ROOT)
print("Rater outputs:", OUTPUT_ROOT)

Mounted at /content/drive
ERM: /content/drive/MyDrive/evidential_alignment_colab/erm_base_models/waterbirds_erm_resnet50_imagenet_100ep_best_val_acc.pt
Embedding cache: /content/drive/MyDrive/evidential_alignment_colab/embeddings_waterbirds_erm
Rater outputs: /content/drive/MyDrive/evidential_alignment_colab/rater_classwise_top_loss_runs


In [2]:
# 2. Fresh repo + dependencies
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone",
    "https://github.com/TheRealMmd/evidential_alignment.git",
    str(REPO_DIR),
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "timm", "transformers", "wilds", "wget", "matplotlib", "scipy",
], check=True)

print("✓ repo ready")

✓ repo ready


In [3]:
# 3. Waterbirds
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not (WB_ARCHIVE.exists() and WB_ARCHIVE.stat().st_size > 100_000_000):
    subprocess.run([
        "wget", "-c",
        "https://downloads.cs.stanford.edu/nlp/data/dro/waterbird_complete95_forest2water2.tar.gz",
        "-O", str(WB_ARCHIVE),
    ], check=True)

if not (WB_DIR / "metadata.csv").exists():
    subprocess.run([
        "tar", "-xzf", str(WB_ARCHIVE), "-C", str(DATA_ROOT)
    ], check=True)

assert (WB_DIR / "metadata.csv").exists()
print("✓ Waterbirds ready")

✓ Waterbirds ready


In [4]:
# 4. Patch data path + PyTorch 2.6 loading
config_path = REPO_DIR / "config.py"
text = config_path.read_text(encoding="utf-8")
placeholder = '"waterbirds": "/path/to/datasets/waterbirds"'
if placeholder in text:
    text = text.replace(placeholder, f'"waterbirds": "{WB_DIR}"')
else:
    text, n = re.subn(
        r'("waterbirds"\s*:\s*)["\'][^"\']*["\']',
        lambda m: m.group(1) + '"' + str(WB_DIR) + '"',
        text,
        count=1,
    )
    assert n == 1
config_path.write_text(text, encoding="utf-8")

algorithm_path = REPO_DIR / "algorithms" / "algorithm.py"
text = algorithm_path.read_text(encoding="utf-8")
text = text.replace(
    'torch.load(check_point, map_location="cpu")',
    'torch.load(check_point, map_location="cpu", weights_only=False)',
)
text = text.replace(
    "torch.load(check_point, map_location='cpu')",
    'torch.load(check_point, map_location="cpu", weights_only=False)',
)
algorithm_path.write_text(text, encoding="utf-8")
print("✓ repo patched")

✓ repo patched


In [5]:
# 5. Install the updated rater.py WITHOUT embedding a huge source string
#
# Recommended:
#   download `rater_classwise_top_loss.py` together with this notebook.
#   On the first run, this cell will ask you to upload it.
#   A small reusable copy is then saved to Drive so future runs are automatic.

import ast
import shutil
from google.colab import files

CODE_DIR = DRIVE_ROOT / "code"
CODE_DIR.mkdir(parents=True, exist_ok=True)

SAVED_RATER_SOURCE = (
    CODE_DIR
    / "rater_classwise_top_loss.py"
)

if SAVED_RATER_SOURCE.exists():
    source_path = SAVED_RATER_SOURCE
    print("Using saved updated rater.py:")
    print(source_path)

else:
    print(
        "Upload the provided file: rater_classwise_top_loss.py"
    )

    uploaded = files.upload()

    candidates = [
        name
        for name in uploaded
        if name.endswith(".py")
    ]

    if len(candidates) != 1:
        raise RuntimeError(
            "Please upload exactly one .py file: "
            "rater_classwise_top_loss.py"
        )

    uploaded_name = candidates[0]
    source_text = uploaded[
        uploaded_name
    ].decode("utf-8")

    SAVED_RATER_SOURCE.write_text(
        source_text,
        encoding="utf-8",
    )

    source_path = SAVED_RATER_SOURCE

    print("Saved reusable copy to:")
    print(source_path)

source_text = source_path.read_text(
    encoding="utf-8"
)

# Safety checks that this is the requested version.
assert "RATER_OUTER_TOP_LOSS_PERCENT" in source_text
assert "_classwise_top_loss_outer_objective" in source_text
assert "Save ONE 5-panel histogram figure" in source_text

ast.parse(source_text)

rater_path = (
    REPO_DIR
    / "algorithms"
    / "rater.py"
)

rater_path.write_text(
    source_text,
    encoding="utf-8",
)

init_path = (
    REPO_DIR
    / "algorithms"
    / "__init__.py"
)

init_text = init_path.read_text(
    encoding="utf-8"
)

if "from . import rater" not in init_text:
    init_path.write_text(
        init_text.rstrip()
        + "\nfrom . import rater\n",
        encoding="utf-8",
    )

print("✓ class-wise top-loss rater.py installed")

Upload the provided file: rater_classwise_top_loss.py


Saving rater_classwise_top_loss.py to rater_classwise_top_loss.py
Saved reusable copy to:
/content/drive/MyDrive/evidential_alignment_colab/code/rater_classwise_top_loss.py
✓ class-wise top-loss rater.py installed


In [6]:
# 6. Experiment parameters
META_STEPS = 500
BATCH_SIZE = 128
NUM_INNER_MODELS = 2

# Every training example receives a Rater-derived weight.
RATER_WEIGHTING = "sigmoid"
INNER_LR = 2e-4
TEMPERATURE = 1.0
OUTER_LR = 3e-4
GRAD_CLIP = 5.0

# ============================================================
# NEW: Rater outer-loss hard-tail percentage PER CLASS
# ============================================================
# Examples:
#   30.0 -> top 30% losses inside class 0 AND top 30% inside class 1
#   20.0 -> top 20% per class
#   50.0 -> top 50% per class
#   100.0 -> all examples per class (class-balanced mean CE)
OUTER_TOP_LOSS_PERCENT = 30.0

# Diagnostics: evaluate often, but avoid hundreds of PNGs.
EVAL_FREQ = 10
PLOT_FREQ = 25

os.environ.update({
    "RATER_ERM_MODEL": str(ERM_MODEL),
    "RATER_EMBEDDING_CACHE_ROOT": str(EMBED_ROOT),
    "RATER_META_STEPS": str(META_STEPS),
    "RATER_NUM_INNER_MODELS": str(NUM_INNER_MODELS),
    "RATER_WEIGHTING": RATER_WEIGHTING,
    "RATER_INNER_LR": str(INNER_LR),
    "RATER_OUTER_LR": str(OUTER_LR),
    "RATER_TEMPERATURE": str(TEMPERATURE),
    "RATER_GRAD_CLIP": str(GRAD_CLIP),
    "RATER_OUTER_TOP_LOSS_PERCENT": str(OUTER_TOP_LOSS_PERCENT),
    "RATER_EVAL_FREQ": str(EVAL_FREQ),
    "RATER_PLOT_FREQ": str(PLOT_FREQ),
})

print({
    "meta_steps": META_STEPS,
    "batch_size": BATCH_SIZE,
    "inner_models": NUM_INNER_MODELS,
    "weighting": RATER_WEIGHTING,
    "inner_lr": INNER_LR,
    "outer_lr": OUTER_LR,
    "temperature": TEMPERATURE,
    "outer_top_loss_percent_per_class": OUTER_TOP_LOSS_PERCENT,
    "outer_uses_group_annotations": False,
})

{'meta_steps': 500, 'batch_size': 128, 'inner_models': 2, 'weighting': 'sigmoid', 'inner_lr': 0.0002, 'outer_lr': 0.0003, 'temperature': 1.0, 'outer_top_loss_percent_per_class': 30.0, 'outer_uses_group_annotations': False}


In [7]:
# 7. Precompute/reuse persistent ERM embeddings
os.environ["RATER_PRECOMPUTE_EMBEDDINGS_ONLY"] = "1"
shutil.rmtree(LOCAL_PRE, ignore_errors=True)
LOCAL_PRE.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", "main.py",
    "--dataset", "waterbirds",
    "--algorithm", "rater",
    "--backbone", "resnet50",
    "--pretrained", "True",
    "--erm_model", str(ERM_MODEL),
    "--save_folder", str(LOCAL_PRE),
    "--train_split", "train",
    "--split_val", "0.4",
    "--mode", "train",
    "--epoch", str(META_STEPS),
]

result = subprocess.run(cmd, cwd=str(REPO_DIR), env=os.environ.copy())
if result.returncode != 0:
    raise RuntimeError("Embedding preparation failed")

shutil.rmtree(LOCAL_PRE, ignore_errors=True)
print("✓ persistent ERM embeddings ready")

✓ persistent ERM embeddings ready


In [8]:
# 8. Verify cache
cache_files = sorted(EMBED_ROOT.rglob("*.pt"))
for p in cache_files:
    print(p.relative_to(EMBED_ROOT), f"{p.stat().st_size / 1024**2:.2f} MB")

assert any(p.name == "train_no_aug.pt" for p in cache_files)
assert any("val_subset1" in p.name for p in cache_files)
assert any("val_subset2" in p.name for p in cache_files)
assert any(p.name == "test.pt" for p in cache_files)

sample = next(p for p in cache_files if p.name == "train_no_aug.pt")
payload = torch.load(sample, map_location="cpu", weights_only=False)
assert payload["cache_metadata"]["representation_source"] == "waterbirds_erm_checkpoint"
print("✓ cache verified:", tuple(payload["embeddings"].shape))

waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/test.pt 45.40 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/train_no_aug.pt 37.57 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val.pt 9.40 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p4_seed_0.pt 3.76 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_0.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_1.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_42.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_63.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_69.pt 5.64 MB
waterbirds/resnet50_water

In [9]:
# 9. Run class-wise top-loss Rater meta-learning
os.environ["RATER_PRECOMPUTE_EMBEDDINGS_ONLY"] = "0"
shutil.rmtree(LOCAL_RUN, ignore_errors=True)
LOCAL_RUN.mkdir(parents=True, exist_ok=True)

# Rater hyperparameters live in environment variables, so the command is short.
cmd = [
    sys.executable, "-u", "main.py",
    "--dataset", "waterbirds",
    "--algorithm", "rater",
    "--backbone", "resnet50",
    "--pretrained", "True",
    "--erm_model", str(ERM_MODEL),
    "--save_folder", str(LOCAL_RUN),
    "--train_split", "train",
    "--test_split", "test",
    "--split_val", "0.4",
    "--batch_size", str(BATCH_SIZE),
    "--mode", "train",
    "--epoch", str(META_STEPS),
    "--seed", "0",
]

result = subprocess.run(cmd, cwd=str(REPO_DIR), env=os.environ.copy())
if result.returncode != 0:
    raise RuntimeError("Rater meta-training failed")

print("✓ Rater meta-training complete")

✓ Rater meta-training complete


In [10]:
# 10. Copy only model + plots + log + summary to Drive
rater_files = sorted(LOCAL_RUN.rglob("final_rater.pt"))
assert len(rater_files) == 1, f"found {len(rater_files)} final_rater.pt files"

local_exp = rater_files[0].parent
for name in ["final_rater.pt", "log.txt", "summary.txt", "plots"]:
    assert (local_exp / name).exists(), name

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_run = OUTPUT_ROOT / (
    f"top{OUTER_TOP_LOSS_PERCENT:g}pct_per_class_all_sample_{stamp}"
)
drive_run.mkdir(parents=True, exist_ok=False)

for name in ["final_rater.pt", "log.txt", "summary.txt"]:
    shutil.copy2(local_exp / name, drive_run / name)
shutil.copytree(local_exp / "plots", drive_run / "plots")

print("Saved:", drive_run)
print("Rater:", drive_run / "final_rater.pt")
print("Plots:", drive_run / "plots")

Saved: /content/drive/MyDrive/evidential_alignment_colab/rater_classwise_top_loss_runs/top30pct_per_class_all_sample_20261007_215941
Rater: /content/drive/MyDrive/evidential_alignment_colab/rater_classwise_top_loss_runs/top30pct_per_class_all_sample_20261007_215941/final_rater.pt
Plots: /content/drive/MyDrive/evidential_alignment_colab/rater_classwise_top_loss_runs/top30pct_per_class_all_sample_20261007_215941/plots


In [11]:
# 11. Summary + plot inventory
print((drive_run / "summary.txt").read_text(encoding="utf-8"))

plots = sorted((drive_run / "plots").rglob("*.png"))
print(f"Saved {len(plots)} PNG plots")
for p in plots:
    print(p.relative_to(drive_run))
print("Outer top-loss percent per class:", OUTER_TOP_LOSS_PERCENT)
print("Outer objective uses group annotations: False")


RATER META-LEARNING SUMMARY

Representation: frozen Waterbirds-ERM ResNet-50 embeddings
ERM checkpoint: /content/drive/MyDrive/evidential_alignment_colab/erm_base_models/waterbirds_erm_resnet50_imagenet_100ep_best_val_acc.pt
ERM fingerprint: 1055b1e8de29
Feature dimension: 2048

Inner models: 2 independent random Linear(2048, 2) classifiers
Initialized from ERM classifier: NO
Reinitialized/refreshed during meta-training: NO
Before every Rater update: BOTH inner models each see the ENTIRE train_no_aug set exactly once

Meta steps: 500
Full inner steps per meta update: 38
Weighting transform for ALL samples: sigmoid
Temperature: 1.0
All samples receive Rater ratings regardless of correctness
Inner LR: 0.0002
Outer LR: 0.0003
Gradient clip: 5.0
Inner objective: SUM_i weight_i * CE_i
Divide by sum(weights): NO
Rater meta objective: class-wise top-loss outer CE; NO group annotations
Outer top-loss percentage per class: 30%
Outer aggregation: mean selected CE within each class, then equal me